# Data Inspection
Fetch raw price data, compare raw vs adjusted prices, inspect what the engine actually sees.

In [20]:
import sys
sys.path.insert(0, '..')  # make trading_engine importable from notebooks/

import yfinance as yf
import pandas as pd
from trading_engine.data.yfinance_loader import YFinanceLoader
from datetime import date

## Raw vs Adjusted prices
The engine uses `yf.download()` which defaults to `auto_adjust=True`.
This adjusts all historical prices for dividends and splits — so `Close` ≠ the price on your broker.

In [ ]:
SYMBOL = "MSFT"
START  = "2020-01-01"
END    = date.today().isoformat()

raw = yf.download(SYMBOL, start=START, end=END, progress=False, auto_adjust=False)
adj = yf.download(SYMBOL, start=START, end=END, progress=False, auto_adjust=True)

for df in [raw, adj]:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

print("=== RAW (what you see on broker) ===")
print(raw[["Open", "High", "Low", "Close", "Adj Close", "Volume"]].tail(5).to_string())

print("\n=== ADJUSTED (what the engine uses) ===")
print(adj[["Open", "High", "Low", "Close", "Volume"]].tail(5).to_string())

In [22]:
# All-time high comparison
raw_ath = raw["Close"].max()
raw_ath_date = raw["Close"].idxmax().date()
adj_ath = adj["Close"].max()
adj_ath_date = adj["Close"].idxmax().date()

print(f"RAW  all-time high: ${raw_ath:.4f} on {raw_ath_date}")
print(f"ADJ  all-time high: ${adj_ath:.4f} on {adj_ath_date}")
print(f"\nRAW  current close: ${raw['Close'].iloc[-1]:.4f} ({raw.index[-1].date()})")
print(f"ADJ  current close: ${adj['Close'].iloc[-1]:.4f} ({adj.index[-1].date()})")

RAW  all-time high: $542.0700 on 2025-10-28
ADJ  all-time high: $539.8253 on 2025-10-28

RAW  current close: $381.8500 (2026-03-20)
ADJ  current close: $381.8500 (2026-03-20)


## Load data via the engine's loader (what the API actually calls)

In [ ]:
loader = YFinanceLoader()
prices = loader.load(SYMBOL, date(2000, 1, 1), date.today())

print(f"Symbol:  {prices.symbol}")
print(f"Source:  {prices.source}")
print(f"Rows:    {len(prices.data)}")
print(f"Columns: {list(prices.data.columns)}")
print()
print(prices.data.tail(5).to_string())

In [24]:
# Summary stats on close price
close = prices.data["close"]
print(f"All-time high (engine data): ${close.max():.4f} on {close.idxmax().date()}")
print(f"All-time low  (engine data): ${close.min():.4f} on {close.idxmin().date()}")
print(f"Current close (engine data): ${close.iloc[-1]:.4f} on {close.index[-1].date()}")
print(f"Total bars: {len(close)}")

All-time high (engine data): $539.8252 on 2025-10-28
All-time low  (engine data): $11.1132 on 2009-03-09
Current close (engine data): $381.8500 on 2026-03-20
Total bars: 6593


## Price Chart (TradingView Lightweight Charts)
Interactive candlestick chart with volume. Uses the engine's adjusted data.
- Zoom: scroll wheel
- Pan: click + drag
- Hover: crosshair with OHLCV tooltip

In [ ]:
from lightweight_charts.widgets import JupyterChart

# ── Config ────────────────────────────────────────────────────────────────────
CHART_SYMBOL = "MSFT"        # ← change this
CHART_START  = date(2020, 1, 1)  # ← change this
CHART_END    = date.today()

# Load via engine loader (adjusted prices)
pf = loader.load(CHART_SYMBOL, CHART_START, CHART_END)

# Reshape to the format lightweight-charts expects
ohlcv = pf.data.copy()
ohlcv.index.name = "time"
ohlcv = ohlcv.reset_index()
ohlcv["time"] = ohlcv["time"].dt.strftime("%Y-%m-%d")

# Build chart
chart = JupyterChart()
chart.layout(background_color="#0f0f10", text_color="#d1d4dc", font_size=12)
chart.candle_style(
    up_color="#26a69a",
    down_color="#ef5350",
    border_up_color="#26a69a",
    border_down_color="#ef5350",
    wick_up_color="#26a69a",
    wick_down_color="#ef5350",
)
chart.volume_config(up_color="#26a69a55", down_color="#ef535055")
chart.crosshair(mode="normal")
chart.time_scale(right_offset=5, min_bar_spacing=2)
chart.watermark(f"{CHART_SYMBOL} · Adjusted Close", color="rgba(255,255,255,0.07)")
chart.topbar.textbox("symbol", f"{CHART_SYMBOL}  |  {CHART_START} → {CHART_END}  |  adj. close")

chart.set(ohlcv)
chart.load()

## Price Chart with Moving Averages
Overlay SMA-50 and SMA-200 on the same chart.

In [ ]:
from lightweight_charts.widgets import JupyterChart

# ── Config ────────────────────────────────────────────────────────────────────
MA_SYMBOL = "MSFT"           # ← change this
MA_START  = date(2020, 1, 1) # ← change this
MA_END    = date.today()
FAST_MA   = 50               # ← change this
SLOW_MA   = 200              # ← change this

pf2 = loader.load(MA_SYMBOL, MA_START, MA_END)
close2 = pf2.data["close"]

# Build OHLCV frame
ohlcv2 = pf2.data.copy()
ohlcv2.index.name = "time"
ohlcv2 = ohlcv2.reset_index()
ohlcv2["time"] = ohlcv2["time"].dt.strftime("%Y-%m-%d")

# Build MA line frames — column must be named "value" (no name passed to create_line)
def to_line_df(series):
    df = series.dropna().reset_index()
    df.columns = ["time", "value"]
    df["time"] = df["time"].dt.strftime("%Y-%m-%d")
    return df

sma_fast = to_line_df(close2.rolling(FAST_MA).mean())
sma_slow = to_line_df(close2.rolling(SLOW_MA).mean())

# Chart
chart2 = JupyterChart()
chart2.layout(background_color="#0f0f10", text_color="#d1d4dc", font_size=12)
chart2.candle_style(
    up_color="#26a69a", down_color="#ef5350",
    border_up_color="#26a69a", border_down_color="#ef5350",
    wick_up_color="#26a69a", wick_down_color="#ef5350",
)
chart2.volume_config(up_color="#26a69a55", down_color="#ef535055")
chart2.watermark(f"{MA_SYMBOL} · SMA{FAST_MA} / SMA{SLOW_MA}", color="rgba(255,255,255,0.07)")

# name="" means the library expects a "value" column (not a named column)
fast_line = chart2.create_line(name="", color="#f5a623", width=1, price_label=True)
slow_line = chart2.create_line(name="", color="#4a90d9", width=1, price_label=True)

chart2.set(ohlcv2)
fast_line.set(sma_fast)
slow_line.set(sma_slow)
chart2.load()